# Camada Gold — Preditor de Bola de Ouro (Arquitetura Medallion)

**Trabalho 1 — Arquitetura Medallion | BigQuery ML**

Este notebook implementa a **camada Gold** do pipeline: parte de `silver.player_season`
(uma linha por jogador-temporada, já limpa e com o target `ballon_dor_score` calculado)
e entrega:

1. a tabela `gold.player_season_features` — features finais normalizadas + target, pronta para o `CREATE MODEL`;
2. os modelos de **BigQuery ML** (`LINEAR_REG` e `BOOSTED_TREE_REGRESSOR`) treinados sobre essa tabela;
3. a avaliação dos modelos (`ML.EVALUATE`, `ML.FEATURE_IMPORTANCE`);
4. a comparação **Top-10 previsto vs. Top-10 real** do Ballon d'Or nas temporadas de teste — o ponto alto da apresentação;
5. a extensão opcional do plano: modelos separados por posição (ataque / meio-campo / defesa+goleiro).

### Etapas implementadas

| # | Etapa | Onde |
|---|-------|------|
| 1 | Configuração e leitura da Silver | §1, §2 |
| 2 | Split treino/teste por temporada (histórico, não vazamento) | §3 |
| 3 | Seleção e normalização de features (z-score por temporada) | §4 |
| 4 | Materialização de `gold.player_season_features` | §5 |
| 5 | `CREATE MODEL` — LINEAR_REG (baseline) | §6 |
| 6 | `CREATE MODEL` — BOOSTED_TREE_REGRESSOR (modelo principal) | §7 |
| 7 | `ML.EVALUATE` e `ML.FEATURE_IMPORTANCE` | §8 |
| 8 | Ranking Top-10 previsto vs. real (demo ao vivo) | §9 |
| 9 | Métrica de acerto do ranking (overlap Top-10 / Top-30) | §10 |
| 10 | Extensão opcional — modelos por posição | §11 |
| 11 | Views de BI para consulta ao vivo na apresentação | §12 |
| 12 | Resumo final da camada Gold | §13 |
| 13 | Apêndice — SQL puro para rodar no BigQuery Console | §14 |

### Decisões de projeto que valem citar na apresentação

- **Split treino/teste:** por ser uma predição sobre temporadas **históricas encerradas**, o split é temporal
  (treina nas temporadas mais antigas, testa nas mais recentes) — nunca aleatório linha-a-linha, o que vazaria
  informação da mesma safra entre treino e teste.
- **Normalização por temporada:** features como "gols/90" têm médias de liga que mudam ano a ano (inflação de gols,
  mudança de regras). Normalizar (z-score) **dentro de cada temporada** deixa o modelo comparando jogadores
  contra seus pares do mesmo ano, e não contra uma média histórica que não existia naquele momento.
- **Dois modelos, um propósito didático:** `LINEAR_REG` como baseline interpretável (coeficientes = direção do efeito)
  e `BOOSTED_TREE_REGRESSOR` como modelo principal (captura não-linearidades e interações entre posição e feature,
  ex.: xG importa muito para atacante, pouco para zagueiro). A apresentação compara os dois.
- **Reamostragem opcional por posição (§11):** três modelos (ataque, meio-campo, defesa/goleiro) em vez de um único
  modelo geral, já que os critérios de destaque do prêmio diferem muito por posição — conforme extensão do plano.


In [ ]:
# --- projeto GCP detectado automaticamente (não precisa editar ao trocar de projeto) ---
import os, subprocess

def get_project_id():
    for var in ("GOOGLE_CLOUD_PROJECT", "GCLOUD_PROJECT", "CLOUDSDK_CORE_PROJECT"):
        if os.environ.get(var):
            return os.environ[var]
    try:
        import google.auth
        _, proj = google.auth.default()
        if proj:
            return proj
    except Exception:
        pass
    try:
        proj = subprocess.run(["gcloud", "config", "get-value", "project"],
                              capture_output=True, text=True).stdout.strip()
        if proj and proj != "(unset)":
            return proj
    except Exception:
        pass
    raise RuntimeError('Projeto não encontrado. Rode: os.environ["GOOGLE_CLOUD_PROJECT"] = "seu-projeto"')

PROJECT_ID = get_project_id()
print("Projeto GCP:", PROJECT_ID)


## 1. Configuração

In [ ]:
from google.cloud import bigquery

PROJECT = PROJECT_ID   # detectado na célula anterior
SILVER  = "silver"
GOLD    = "gold"
LOCATION = "US"

client = bigquery.Client(project=PROJECT)

# garante que o dataset gold existe (bronze e silver já foram criados no notebook de ingestão)
ds = bigquery.Dataset(f"{PROJECT}.{GOLD}")
ds.location = LOCATION
ds.description = "Dados agregados e prontos para ML e BI"
client.create_dataset(ds, exists_ok=True)

print(f"Projeto : {PROJECT}")
print(f"Silver  : {PROJECT}.{SILVER}")
print(f"Gold    : {PROJECT}.{GOLD}")


Projeto : trabalho-pdm-joao-fut
Silver  : trabalho-pdm-joao-fut.silver
Gold    : trabalho-pdm-joao-fut.gold


In [ ]:
import pandas as pd


def bq(sql):
    """Executa uma query e retorna DataFrame."""
    return client.query(sql).to_dataframe()


def bq_run(sql):
    """Executa DDL/DML (CREATE MODEL, CREATE TABLE etc.) e espera terminar."""
    job = client.query(sql)
    job.result()
    return job


## 2. Leitura e inspeção da camada Silver

Confirma que `silver.player_season` existe e tem o formato esperado antes de construir a Gold em cima dela.

In [ ]:
t = client.get_table(f"{PROJECT}.{SILVER}.player_season")
print(f"silver.player_season: {t.num_rows:,} linhas | {len(t.schema)} colunas")

preview = bq(f"""
    SELECT season_label, league, COUNT(*) AS jogadores,
           SUM(is_ballon_dor_ranked) AS ranqueados,
           MAX(ballon_dor_score) AS score_max
    FROM `{PROJECT}.{SILVER}.player_season`
    GROUP BY season_label, league
    ORDER BY season_label, league
""")
display(preview)


silver.player_season: 12,917 linhas | 101 colunas


,season_label,league,jogadores,ranqueados,score_max
0,2017-18,ENG-Premier League,332,11,24.9668
1,2017-18,ESP-La Liga,350,13,100.0000
2,2017-18,FRA-Ligue 1,332,3,46.0823
3,2017-18,GER-Bundesliga,293,0,0.0000
4,2017-18,ITA-Serie A,330,2,0.2656
5,2018-19,ENG-Premier League,322,16,98.9796
6,2018-19,ESP-La Liga,329,4,100.0000
7,2018-19,FRA-Ligue 1,330,2,12.9738
8,2018-19,GER-Bundesliga,288,1,6.4140
9,2018-19,ITA-Serie A,338,2,69.3878


In [ ]:
# temporadas com target disponível (rotuladas com o Ballon d'Or) — só estas entram no treino/avaliação
temporadas = bq(f"""
    SELECT DISTINCT season, season_label, has_target
    FROM `{PROJECT}.{SILVER}.player_season`
    ORDER BY season
""")
display(temporadas)

SEASONS_ROTULADAS = sorted(temporadas.query("has_target == 1").season.tolist())
print("\ntemporadas rotuladas (com target):", SEASONS_ROTULADAS)


,season,season_label,has_target
0,1718,2017-18,1
1,1819,2018-19,1
2,1920,2019-20,0
3,2021,2020-21,1
4,2122,2021-22,1
5,2223,2022-23,1
6,2324,2023-24,1
7,2425,2024-25,0



temporadas rotuladas (com target): [1718, 1819, 2021, 2122, 2223, 2324]


/tmp/ipykernel_114/4283982815.py:9: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  SEASONS_ROTULADAS = sorted(temporadas.query("has_target == 1").season.tolist())


## 3. Split treino/teste — temporal, não aleatório

Como o objetivo é prever um prêmio anual, o split correto é **por temporada**, nunca por linha aleatória:
treina nas temporadas mais antigas e testa nas mais recentes, simulando a situação real de "prever o próximo ano
com dados só do passado". Isso evita vazamento sutil de informação da mesma safra entre treino e teste.

Regra: as **duas temporadas rotuladas mais recentes** viram teste (para ter massa suficiente de Top-10 real
para comparar); o restante vira treino. Ajustável em `N_TEMPORADAS_TESTE`.

In [ ]:
N_TEMPORADAS_TESTE = 2  # últimas N temporadas rotuladas viram teste

SEASONS_TESTE  = SEASONS_ROTULADAS[-N_TEMPORADAS_TESTE:]
SEASONS_TREINO = [s for s in SEASONS_ROTULADAS if s not in SEASONS_TESTE]

print("treino:", SEASONS_TREINO)
print("teste :", SEASONS_TESTE)

# jogador-temporada sem target (fora do escopo rotulado, ex. 2024-25) fica de fora do treino/teste
# mas continua disponível na Gold para inferência/demo se o grupo quiser prever a temporada corrente
SEASONS_INFERENCIA = sorted(temporadas.query("has_target == 0").season.tolist())
print("sem rótulo (só para inferência, se desejado):", SEASONS_INFERENCIA)


treino: [1718, 1819, 2021, 2122]
teste : [2223, 2324]
sem rótulo (só para inferência, se desejado): [1920, 2425]


/tmp/ipykernel_114/203700799.py:11: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  SEASONS_INFERENCIA = sorted(temporadas.query("has_target == 0").season.tolist())


## 4. Seleção e normalização de features

A Silver já tem ~50 colunas de stats. A Gold seleciona o subconjunto que entra no modelo e aplica
**z-score por temporada**: cada feature numérica é centralizada e escalada usando a média/desvio-padrão
**daquela mesma temporada**, não do histórico inteiro. Isso deixa "2.1 gols/90" comparável entre 2018 e 2024
mesmo que o nível geral de gols tenha mudado.

Features de contexto de time (`is_league_champion_proxy`, `is_ucl_spot_proxy` etc.) e categóricas
(`pos_primary`, `league`) **não** são normalizadas — são binárias/categóricas por natureza.

O `BOOSTED_TREE_REGRESSOR` do BigQuery ML já lida bem com escalas diferentes (é baseado em árvores),
mas o `LINEAR_REG` se beneficia da normalização — por isso ela é feita uma vez na Gold e serve os dois modelos.

In [ ]:
FEATURES_NUM = [
    # finalização
    "goals_per90", "npg_per90", "xg_per90", "npxg_per90", "shots_per90", "sot_per90",
    "sot_pct", "g_minus_xg",
    # criação
    "assists_per90", "xag_per90", "sca_per90", "gca_per90", "kp_per90",
    "ga_per90", "xg_xag_per90", "pass_cmp_pct",
    # progressão / condução
    "prgc_per90", "prgp_per90", "prgr_per90", "take_on_succ_pct", "touches_att_pen_per90",
    # defesa
    "tkl_per90", "int_per90", "clr_per90", "blocks_per90", "recov_per90", "aerial_won_pct",
    # volume / contexto individual
    "minutes", "age",
    # contexto de time
    "team_ppm", "team_ppm_on_pitch",
]

FEATURES_CTX = [
    "is_league_champion_proxy", "is_ucl_spot_proxy", "is_europe_spot_proxy", "is_gk",
]

FEATURES_CAT = ["pos_primary", "league"]

# confere que todas as colunas esperadas existem na Silver antes de montar o SQL
cols_silver = set(f.name for f in t.schema)
faltando = [c for c in FEATURES_NUM + FEATURES_CTX + FEATURES_CAT if c not in cols_silver]
print("features numéricas  :", len(FEATURES_NUM))
print("features contexto   :", len(FEATURES_CTX))
print("features categóricas:", len(FEATURES_CAT))
print("faltando na Silver  :", faltando if faltando else "nenhuma — pode seguir")


features numéricas  : 31
features contexto   : 4
features categóricas: 2
faltando na Silver  : nenhuma — pode seguir


In [ ]:
# SQL da normalização z-score por temporada, gerado dinamicamente a partir da lista de features
norm_exprs = ",\n    ".join(
    f"SAFE_DIVIDE(({c} - AVG({c}) OVER (PARTITION BY season)), "
    f"NULLIF(STDDEV({c}) OVER (PARTITION BY season), 0)) AS {c}_z"
    for c in FEATURES_NUM
)
print(norm_exprs[:600], "...\n")
print(f"total de expressões de normalização geradas: {len(FEATURES_NUM)}")


SAFE_DIVIDE((goals_per90 - AVG(goals_per90) OVER (PARTITION BY season)), NULLIF(STDDEV(goals_per90) OVER (PARTITION BY season), 0)) AS goals_per90_z,
    SAFE_DIVIDE((npg_per90 - AVG(npg_per90) OVER (PARTITION BY season)), NULLIF(STDDEV(npg_per90) OVER (PARTITION BY season), 0)) AS npg_per90_z,
    SAFE_DIVIDE((xg_per90 - AVG(xg_per90) OVER (PARTITION BY season)), NULLIF(STDDEV(xg_per90) OVER (PARTITION BY season), 0)) AS xg_per90_z,
    SAFE_DIVIDE((npxg_per90 - AVG(npxg_per90) OVER (PARTITION BY season)), NULLIF(STDDEV(npxg_per90) OVER (PARTITION BY season), 0)) AS npxg_per90_z,
    SAFE_DIV ...

total de expressões de normalização geradas: 31


## 5. Materialização de `gold.player_season_features`

Uma linha por jogador-temporada, com:

- chave e metadados de identificação (para rastrear de volta ao jogador real na apresentação);
- todas as features normalizadas (`_z`) + as originais em escala natural (para leitura/BI);
- coluna `split` (`train` / `test` / `inference`) já calculada, para o `CREATE MODEL` filtrar direto;
- o target `ballon_dor_score` e as colunas de apoio (`bd_rank`, `is_top10`, `is_winner`).

Nenhuma transformação adicional é necessária depois disso — a tabela vai direto para o `CREATE MODEL`.

In [ ]:
SEASONS_TREINO_SQL = ", ".join(str(s) for s in SEASONS_TREINO)
SEASONS_TESTE_SQL  = ", ".join(str(s) for s in SEASONS_TESTE)

create_gold_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT}.{GOLD}.player_season_features`
CLUSTER BY season, league AS
SELECT
  -- identificacao ---------------------------------------------------------
  season, season_label, award_year, league, player_key, player, club_main,
  pos_primary, pos_raw,

  -- split temporal ----------------------------------------------------------
  CASE
    WHEN season IN ({SEASONS_TREINO_SQL}) THEN 'train'
    WHEN season IN ({SEASONS_TESTE_SQL})  THEN 'test'
    ELSE 'inference'
  END AS split,

  -- features de contexto (nao normalizadas -- ja sao binarias) --------------
  is_league_champion_proxy, is_ucl_spot_proxy, is_europe_spot_proxy,
  team_rank_proxy,

  -- features numericas normalizadas por temporada (z-score) -----------------
  {norm_exprs},

  -- features numericas em escala natural (leitura humana / BI) --------------
  is_gk,
  {", ".join(FEATURES_NUM)},

  -- target e colunas de apoio ------------------------------------------------
  has_target, bd_rank, bd_points, ballon_dor_score, is_ballon_dor_ranked, is_top10, is_winner

FROM `{PROJECT}.{SILVER}.player_season`
"""

bq_run(create_gold_sql)

gold_tbl = client.get_table(f"{PROJECT}.{GOLD}.player_season_features")
print(f"gold.player_season_features: {gold_tbl.num_rows:,} linhas | {len(gold_tbl.schema)} colunas")

gold.player_season_features: 12,917 linhas | 84 colunas


In [ ]:
display(bq(f"""
    SELECT split, season_label, COUNT(*) AS jogadores, SUM(is_ballon_dor_ranked) AS ranqueados
    FROM `{PROJECT}.{GOLD}.player_season_features`
    GROUP BY split, season_label
    ORDER BY split, season_label
"""))


,split,season_label,jogadores,ranqueados
0,inference,2019-20,1575,0
1,inference,2024-25,1576,0
2,test,2022-23,1639,30
3,test,2023-24,1614,30
4,train,2017-18,1637,29
5,train,2018-19,1607,25
6,train,2020-21,1636,30
7,train,2021-22,1633,28


In [ ]:
# ---- contrato da camada Gold ------------------------------------------
checks = bq(f"""
    SELECT
      COUNT(*) AS linhas,
      COUNTIF(ballon_dor_score IS NULL) AS target_nulo,
      COUNTIF(split = 'train' AND has_target = 0) AS treino_sem_target,   -- deve ser 0
      COUNTIF(split = 'test'  AND has_target = 0) AS teste_sem_target,    -- deve ser 0
      COUNT(DISTINCT CONCAT(CAST(season AS STRING), '|', player_key)) AS chaves_distintas,
      COUNT(*) - COUNT(DISTINCT CONCAT(CAST(season AS STRING), '|', player_key)) AS duplicatas
    FROM `{PROJECT}.{GOLD}.player_season_features`
""")
display(checks)

row = checks.iloc[0]
assert row.target_nulo == 0, "target nulo na Gold"
assert row.treino_sem_target == 0, "linha de treino sem target"
assert row.teste_sem_target == 0, "linha de teste sem target"
assert row.duplicatas == 0, "chave duplicada na Gold"
print("todas as validacoes da Gold passaram")


,linhas,target_nulo,treino_sem_target,teste_sem_target,chaves_distintas,duplicatas
0,12917,0,0,0,12917,0


todas as validacoes da Gold passaram


## 6. `CREATE MODEL` — LINEAR_REG (baseline)

Modelo linear como baseline interpretável: os coeficientes mostram diretamente a direção e a magnitude
do efeito de cada feature no score. Treinado **só** com `split = 'train'`.

Features categóricas (`pos_primary`, `league`) entram como estão — o BigQuery ML faz one-hot encoding
automático de colunas `STRING` em `CREATE MODEL`.

In [ ]:
features_z = [f"{c}_z" for c in FEATURES_NUM]
todas_features = features_z + FEATURES_CTX + FEATURES_CAT
features_select = ",\n  ".join(todas_features)

create_linreg_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT}.{GOLD}.model_linreg_ballon_dor`
OPTIONS(
  model_type = 'LINEAR_REG',
  input_label_cols = ['ballon_dor_score'],
  data_split_method = 'NO_SPLIT',   -- o split ja foi feito na Gold (coluna `split`)
  l2_reg = 1.0
) AS
SELECT
  {features_select},
  ballon_dor_score
FROM `{PROJECT}.{GOLD}.player_season_features`
WHERE split = 'train'
"""

print(create_linreg_sql)
bq_run(create_linreg_sql)
print("\nmodel_linreg_ballon_dor treinado.")



CREATE OR REPLACE MODEL `trabalho-pdm-joao-fut.gold.model_linreg_ballon_dor`
OPTIONS(
  model_type = 'LINEAR_REG',
  input_label_cols = ['ballon_dor_score'],
  data_split_method = 'NO_SPLIT',   -- o split ja foi feito na Gold (coluna `split`)
  l2_reg = 1.0
) AS
SELECT
  goals_per90_z,
  npg_per90_z,
  xg_per90_z,
  npxg_per90_z,
  shots_per90_z,
  sot_per90_z,
  sot_pct_z,
  g_minus_xg_z,
  assists_per90_z,
  xag_per90_z,
  sca_per90_z,
  gca_per90_z,
  kp_per90_z,
  ga_per90_z,
  xg_xag_per90_z,
  pass_cmp_pct_z,
  prgc_per90_z,
  prgp_per90_z,
  prgr_per90_z,
  take_on_succ_pct_z,
  touches_att_pen_per90_z,
  tkl_per90_z,
  int_per90_z,
  clr_per90_z,
  blocks_per90_z,
  recov_per90_z,
  aerial_won_pct_z,
  minutes_z,
  age_z,
  team_ppm_z,
  team_ppm_on_pitch_z,
  is_league_champion_proxy,
  is_ucl_spot_proxy,
  is_europe_spot_proxy,
  is_gk,
  pos_primary,
  league,
  ballon_dor_score
FROM `trabalho-pdm-joao-fut.gold.player_season_features`
WHERE split = 'train'


model_linreg_ba

## 7. `CREATE MODEL` — BOOSTED_TREE_REGRESSOR (modelo principal)

Modelo de árvores de gradient boosting (XGBoost sob o capô do BigQuery ML): captura não-linearidades
e interações entre features (ex.: xG/90 importa muito para atacante e quase nada para zagueiro,
algo que o modelo linear não representa bem sem interações manuais).

Hiperparâmetros conservadores para um dataset de poucos milhares de linhas — evitar overfitting
é mais importante aqui do que espremer a última fração de R².

In [ ]:
create_btree_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT}.{GOLD}.model_btree_ballon_dor`
OPTIONS(
  model_type = 'BOOSTED_TREE_REGRESSOR',
  input_label_cols = ['ballon_dor_score'],
  data_split_method = 'NO_SPLIT',
  max_iterations = 60,
  learn_rate = 0.08,
  max_tree_depth = 6,
  subsample = 0.85,
  l2_reg = 1.0,
  early_stop = TRUE,
  min_rel_progress = 0.001
) AS
SELECT
  {features_select},
  ballon_dor_score
FROM `{PROJECT}.{GOLD}.player_season_features`
WHERE split = 'train'
"""

print(create_btree_sql)
bq_run(create_btree_sql)
print("\nmodel_btree_ballon_dor treinado.")



CREATE OR REPLACE MODEL `trabalho-pdm-joao-fut.gold.model_btree_ballon_dor`
OPTIONS(
  model_type = 'BOOSTED_TREE_REGRESSOR',
  input_label_cols = ['ballon_dor_score'],
  data_split_method = 'NO_SPLIT',
  max_iterations = 60,
  learn_rate = 0.08,
  max_tree_depth = 6,
  subsample = 0.85,
  l2_reg = 1.0,
  early_stop = TRUE,
  min_rel_progress = 0.001
) AS
SELECT
  goals_per90_z,
  npg_per90_z,
  xg_per90_z,
  npxg_per90_z,
  shots_per90_z,
  sot_per90_z,
  sot_pct_z,
  g_minus_xg_z,
  assists_per90_z,
  xag_per90_z,
  sca_per90_z,
  gca_per90_z,
  kp_per90_z,
  ga_per90_z,
  xg_xag_per90_z,
  pass_cmp_pct_z,
  prgc_per90_z,
  prgp_per90_z,
  prgr_per90_z,
  take_on_succ_pct_z,
  touches_att_pen_per90_z,
  tkl_per90_z,
  int_per90_z,
  clr_per90_z,
  blocks_per90_z,
  recov_per90_z,
  aerial_won_pct_z,
  minutes_z,
  age_z,
  team_ppm_z,
  team_ppm_on_pitch_z,
  is_league_champion_proxy,
  is_ucl_spot_proxy,
  is_europe_spot_proxy,
  is_gk,
  pos_primary,
  league,
  ballon_dor_score
F

## 8. Avaliação — `ML.EVALUATE` e `ML.FEATURE_IMPORTANCE`

Avaliação feita **fora da amostra de treino**, passando explicitamente `split = 'test'` para o `ML.EVALUATE`.
Isso é o que dá validade à comparação Top-10 da §9 — sem isso, estaríamos avaliando o modelo nos mesmos
dados em que ele aprendeu.

In [ ]:
eval_linreg = bq(f"""
    SELECT 'LINEAR_REG' AS modelo, *
    FROM ML.EVALUATE(MODEL `{PROJECT}.{GOLD}.model_linreg_ballon_dor`,
      (SELECT {features_select}, ballon_dor_score
       FROM `{PROJECT}.{GOLD}.player_season_features`
       WHERE split = 'test'))
""")

eval_btree = bq(f"""
    SELECT 'BOOSTED_TREE_REGRESSOR' AS modelo, *
    FROM ML.EVALUATE(MODEL `{PROJECT}.{GOLD}.model_btree_ballon_dor`,
      (SELECT {features_select}, ballon_dor_score
       FROM `{PROJECT}.{GOLD}.player_season_features`
       WHERE split = 'test'))
""")

comparacao = pd.concat([eval_linreg, eval_btree], ignore_index=True)
display(comparacao[["modelo", "mean_absolute_error", "mean_squared_error",
                     "median_absolute_error", "r2_score", "explained_variance"]])


,modelo,mean_absolute_error,mean_squared_error,median_absolute_error,r2_score,explained_variance
0,LINEAR_REG,0.990710,15.135288,0.528911,0.087442,0.087523
1,BOOSTED_TREE_REGRESSOR,0.391534,13.107757,0.006422,0.209689,0.210005


In [ ]:
# importancia de features do modelo de arvores -- mostra o que o modelo realmente usa
fi = bq(f"""
    SELECT *
    FROM ML.FEATURE_IMPORTANCE(MODEL `{PROJECT}.{GOLD}.model_btree_ballon_dor`)
    ORDER BY importance_gain DESC
    LIMIT 15
""")
display(fi)


,feature,importance_weight,importance_gain,importance_cover
0,age_z,54,1139.554611,165.759259
1,ga_per90_z,31,1003.922043,5219.774194
2,prgc_per90_z,36,730.891080,36.500000
3,goals_per90_z,148,672.456690,1945.027027
4,clr_per90_z,62,667.572441,166.887097
5,g_minus_xg_z,61,611.241934,544.278689
6,prgp_per90_z,87,559.988515,2090.919540
7,int_per90_z,50,469.844522,260.460000
8,recov_per90_z,33,443.731493,684.424242
9,shots_per90_z,43,399.264493,928.232558


In [ ]:
chart_display = fi.sort_values("importance_gain", ascending=True)
print("Top features por importancia (gain) -- BOOSTED_TREE_REGRESSOR:")
for _, r in chart_display.tail(15).iterrows():
    barra = "#" * int(r.importance_gain / chart_display.importance_gain.max() * 40)
    print(f"  {r.feature:<28} {barra} {r.importance_gain:.3f}")


Top features por importancia (gain) -- BOOSTED_TREE_REGRESSOR:
  prgr_per90_z                 ####### 210.786
  sot_pct_z                    ####### 210.843
  blocks_per90_z               ######### 266.707
  minutes_z                    ########## 295.904
  pass_cmp_pct_z               ############# 386.139
  shots_per90_z                ############## 399.264
  recov_per90_z                ############### 443.731
  int_per90_z                  ################ 469.845
  prgp_per90_z                 ################### 559.989
  g_minus_xg_z                 ##################### 611.242
  clr_per90_z                  ####################### 667.572
  goals_per90_z                ####################### 672.457
  prgc_per90_z                 ######################### 730.891
  ga_per90_z                   ################################### 1003.922
  age_z                        ######################################## 1139.555


## 9. Ranking Top-10 previsto vs. real — o ponto alto da apresentação

Usa `ML.PREDICT` sobre as temporadas de teste com o modelo `BOOSTED_TREE_REGRESSOR` (melhor desempenho
esperado por capturar não-linearidades), monta o ranking Top-10 previsto por temporada e compara lado a lado
com o Top-10 real do prêmio.

In [ ]:
predicoes_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT}.{GOLD}.predictions_test` AS
SELECT
  season, season_label, league, player, club_main, pos_primary,
  bd_rank AS rank_real, ballon_dor_score AS score_real,
  predicted_ballon_dor_score AS score_previsto
FROM ML.PREDICT(MODEL `{PROJECT}.{GOLD}.model_btree_ballon_dor`,
  (SELECT {features_select}, season, season_label, player, club_main,
          bd_rank, ballon_dor_score
   FROM `{PROJECT}.{GOLD}.player_season_features`
   WHERE split = 'test'))
"""
bq_run(predicoes_sql)
print("gold.predictions_test materializada.")


gold.predictions_test materializada.


In [ ]:
for season in SEASONS_TESTE:
    season_label = temporadas.query("season == @season").season_label.iloc[0]

    top10_previsto = bq(f"""
        SELECT player, club_main, pos_primary, ROUND(score_previsto, 2) AS score_previsto,
               rank_real
        FROM `{PROJECT}.{GOLD}.predictions_test`
        WHERE season = {season}
        ORDER BY score_previsto DESC
        LIMIT 10
    """)
    top10_previsto.insert(0, "posicao_prevista", range(1, len(top10_previsto) + 1))

    top10_real = bq(f"""
        SELECT bd_rank AS posicao_real, player, club_main, pos_primary,
               ROUND(ballon_dor_score, 2) AS score_real
        FROM `{PROJECT}.{GOLD}.player_season_features`
        WHERE season = {season} AND bd_rank BETWEEN 1 AND 10
        ORDER BY bd_rank
    """)

    acertos = set(top10_previsto.player) & set(top10_real.player)

    print("=" * 78)
    print(f"TEMPORADA {season_label} -- Bola de Ouro {season % 100 + 2000}")
    print("=" * 78)
    print("\nTop-10 PREVISTO pelo modelo:")
    display(top10_previsto)
    print("\nTop-10 REAL do premio:")
    display(top10_real)
    print(f"\n>>> acertos (jogador no Top-10 previsto E no Top-10 real): {len(acertos)}/10")
    print(f">>> jogadores: {sorted(acertos) if acertos else '(nenhum)'}\n")


/tmp/ipykernel_114/2209965460.py:2: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  season_label = temporadas.query("season == @season").season_label.iloc[0]


TEMPORADA 2022-23 -- Bola de Ouro 2023

Top-10 PREVISTO pelo modelo:


,posicao_prevista,player,club_main,pos_primary,score_previsto,rank_real
0,1,Lionel Messi,Paris S-G,FW,55.55,1.0
1,2,Karim Benzema,Real Madrid,FW,49.34,16.0
2,3,Robert Lewandowski,Barcelona,FW,26.94,12.0
3,4,Kylian Mbappé,Paris S-G,FW,15.47,3.0
4,5,Eric Maxim Choupo-Moting,Bayern Munich,FW,14.57,NaN
5,6,Roberto Firmino,Liverpool,FW,12.19,NaN
6,7,Neymar,Paris S-G,MF,11.06,NaN
7,8,Gabriel Magalhães,Arsenal,DF,9.19,NaN
8,9,Kevin Danso,Lens,DF,7.11,NaN
9,10,Loïs Openda,Lens,FW,6.98,NaN



Top-10 REAL do premio:


,posicao_real,player,club_main,pos_primary,score_real
0,1.0,Lionel Messi,Paris S-G,FW,100.00
1,2.0,Erling Haaland,Manchester City,FW,77.27
2,3.0,Kylian Mbappé,Paris S-G,FW,58.44
3,4.0,Kevin De Bruyne,Manchester City,MF,21.64
4,5.0,Rodri,Manchester City,MF,12.34
5,6.0,Vinicius Júnior,Real Madrid,FW,10.61
6,7.0,Julián Álvarez,Manchester City,FW,6.06
7,8.0,Victor Osimhen,Napoli,FW,5.19
8,9.0,Bernardo Silva,Manchester City,MF,4.33
9,10.0,Luka Modrić,Real Madrid,MF,4.11



>>> acertos (jogador no Top-10 previsto E no Top-10 real): 2/10
>>> jogadores: ['Kylian Mbappé', 'Lionel Messi']



... (2 outputs left out)

,posicao_prevista,player,club_main,pos_primary,score_previsto,rank_real
0,1,Mohamed Salah,Liverpool,FW,29.55,NaN
1,2,Bukayo Saka,Arsenal,FW,16.30,21.0
2,3,Jude Bellingham,Real Madrid,MF,15.02,3.0
3,4,Cole Palmer,Chelsea,FW,12.54,25.0
4,5,Kylian Mbappé,Paris S-G,FW,12.49,6.0
5,6,James Maddison,Tottenham,MF,12.15,NaN
6,7,Victor Boniface,Leverkusen,FW,11.28,NaN
7,8,Kevin De Bruyne,Manchester City,MF,9.87,NaN
8,9,Rodri,Manchester City,MF,9.39,1.0
9,10,Virgil van Dijk,Liverpool,DF,8.39,NaN



Top-10 REAL do premio:


,posicao_real,player,club_main,pos_primary,score_real
0,1.0,Rodri,Manchester City,MF,100.00
1,2.0,Vinicius Júnior,Real Madrid,FW,96.50
2,3.0,Jude Bellingham,Real Madrid,MF,78.38
3,4.0,Dani Carvajal,Real Madrid,DF,47.01
4,5.0,Erling Haaland,Manchester City,FW,36.92
5,6.0,Kylian Mbappé,Paris S-G,FW,35.90
6,7.0,Lautaro Martínez,Inter,FW,34.36
7,8.0,Lamine Yamal,Barcelona,FW,32.73
8,9.0,Toni Kroos,Real Madrid,MF,24.87
9,10.0,Harry Kane,Bayern Munich,FW,17.18



>>> acertos (jogador no Top-10 previsto E no Top-10 real): 3/10
>>> jogadores: ['Jude Bellingham', 'Kylian Mbappé', 'Rodri']



## 10. Métrica de acerto — overlap Top-10 e Top-30

Consolida a comparação da §9 em uma métrica única por temporada, e também numa janela mais generosa
(Top-30, o "shortlist" oficial do prêmio) — útil porque o modelo pode acertar a **região** do ranking
mesmo sem acertar a ordem exata dos 10 primeiros.

In [ ]:
linhas = []
for season in SEASONS_TESTE:
    season_label = temporadas.query("season == @season").season_label.iloc[0]

    prev10 = set(bq(f"""
        SELECT player FROM `{PROJECT}.{GOLD}.predictions_test`
        WHERE season = {season} ORDER BY score_previsto DESC LIMIT 10
    """).player)
    real10 = set(bq(f"""
        SELECT player FROM `{PROJECT}.{GOLD}.player_season_features`
        WHERE season = {season} AND bd_rank BETWEEN 1 AND 10
    """).player)

    prev30 = set(bq(f"""
        SELECT player FROM `{PROJECT}.{GOLD}.predictions_test`
        WHERE season = {season} ORDER BY score_previsto DESC LIMIT 30
    """).player)
    real30 = set(bq(f"""
        SELECT player FROM `{PROJECT}.{GOLD}.player_season_features`
        WHERE season = {season} AND bd_rank BETWEEN 1 AND 30
    """).player)

    vencedor_real = bq(f"""
        SELECT player FROM `{PROJECT}.{GOLD}.player_season_features`
        WHERE season = {season} AND bd_rank = 1
    """).player.iloc[0]

    linhas.append({
        "temporada": season_label,
        "overlap_top10": f"{len(prev10 & real10)}/10",
        "overlap_top30": f"{len(prev30 & real30)}/30",
        "vencedor_real_no_top10_previsto": vencedor_real in prev10,
    })

resumo_ranking = pd.DataFrame(linhas)
display(resumo_ranking)


/tmp/ipykernel_114/1124505573.py:3: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  season_label = temporadas.query("season == @season").season_label.iloc[0]
/tmp/ipykernel_114/1124505573.py:3: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  season_label = temporadas.query("season == @season").season_label.iloc[0]


,temporada,overlap_top10,overlap_top30,vencedor_real_no_top10_previsto
0,2022-23,2/10,10/30,True
1,2023-24,3/10,11/30,True


## 11. Extensão opcional — modelos separados por posição

Conforme a extensão do plano: os critérios de destaque diferem muito entre atacantes, meio-campistas e
defensores/goleiros (um zagueiro nunca vai competir em gols/90 com um atacante, mas pode se destacar em
desarmes e interceptações). Três modelos `BOOSTED_TREE_REGRESSOR` menores, um por grupo de posição.

Esta seção é **opcional** — só rode se o grupo tiver tempo, conforme o plano. O modelo geral (§7) já
cobre o MVP.

In [ ]:
GRUPOS_POSICAO = {
    "atk": ["FW"],
    "mid": ["MF"],
    "def": ["DF", "GK"],
}

pos_case = "CASE " + " ".join(
    f"WHEN pos_primary IN ({', '.join(repr(p) for p in pos_list)}) THEN '{grupo}'"
    for grupo, pos_list in GRUPOS_POSICAO.items()
) + " ELSE 'outro' END"

distrib_pos = bq(f"""
    SELECT {pos_case} AS grupo_posicao, COUNT(*) AS jogadores
    FROM `{PROJECT}.{GOLD}.player_season_features`
    WHERE split = 'train'
    GROUP BY grupo_posicao
    ORDER BY jogadores DESC
""")
display(distrib_pos)


,grupo_posicao,jogadores
0,def,3016
1,mid,2110
2,atk,1387


In [ ]:
features_sem_posicao = [f for f in todas_features if f not in ("pos_primary", "is_gk")]
features_pos_select = ",\n  ".join(features_sem_posicao)

for grupo, pos_list in GRUPOS_POSICAO.items():
    pos_filter = ", ".join(repr(p) for p in pos_list)
    model_name = f"model_btree_{grupo}"

    sql = f"""
    CREATE OR REPLACE MODEL `{PROJECT}.{GOLD}.{model_name}`
    OPTIONS(
      model_type = 'BOOSTED_TREE_REGRESSOR',
      input_label_cols = ['ballon_dor_score'],
      data_split_method = 'NO_SPLIT',
      max_iterations = 50,
      learn_rate = 0.08,
      max_tree_depth = 5,
      subsample = 0.85,
      l2_reg = 1.0,
      early_stop = TRUE
    ) AS
    SELECT
      {features_pos_select},
      ballon_dor_score
    FROM `{PROJECT}.{GOLD}.player_season_features`
    WHERE split = 'train' AND pos_primary IN ({pos_filter})
    """
    bq_run(sql)
    print(f"{model_name} treinado (posicoes: {pos_list}).")


model_btree_mid treinado (posicoes: ['MF']).
model_btree_def treinado (posicoes: ['DF', 'GK']).


In [ ]:
# avaliacao rapida dos 3 modelos por posicao, fora da amostra
linhas_pos = []
for grupo, pos_list in GRUPOS_POSICAO.items():
    pos_filter = ", ".join(repr(p) for p in pos_list)
    model_name = f"model_btree_{grupo}"
    ev = bq(f"""
        SELECT '{grupo}' AS grupo, *
        FROM ML.EVALUATE(MODEL `{PROJECT}.{GOLD}.{model_name}`,
          (SELECT {features_pos_select}, ballon_dor_score
           FROM `{PROJECT}.{GOLD}.player_season_features`
           WHERE split = 'test' AND pos_primary IN ({pos_filter})))
    """)
    linhas_pos.append(ev)

eval_por_posicao = pd.concat(linhas_pos, ignore_index=True)
display(eval_por_posicao[["grupo", "mean_absolute_error", "r2_score", "explained_variance"]])


,grupo,mean_absolute_error,r2_score,explained_variance
0,atk,1.032889,0.088646,0.089204
1,mid,0.405170,-0.000518,0.000095
2,def,0.376122,-0.076810,-0.011888


## 12. Views de BI — consulta ao vivo na apresentação

Duas views simples sobre a Gold, pensadas para rodar **ao vivo** durante a apresentação (o entregável do
plano exige "consulta ao vivo às camadas Medallion e ao modelo de ML"):

- `gold.vw_top10_por_temporada` — ranking previsto vs. real lado a lado, para qualquer temporada de teste;
- `gold.vw_ranking_atual` — aplica o modelo às temporadas **sem rótulo** (`split = 'inference'`, ex. 2024-25),
  útil para a pergunta óbvia da banca: "e se vocês rodarem isso pra hoje?".

In [ ]:
view_top10_sql = f"""
CREATE OR REPLACE VIEW `{PROJECT}.{GOLD}.vw_top10_por_temporada` AS
SELECT
  season, season_label, league, player, club_main, pos_primary,
  rank_real, ROUND(score_real, 2) AS score_real,
  ROUND(score_previsto, 2) AS score_previsto,
  RANK() OVER (PARTITION BY season ORDER BY score_previsto DESC) AS posicao_prevista
FROM `{PROJECT}.{GOLD}.predictions_test`
QUALIFY posicao_prevista <= 10 OR rank_real <= 10
ORDER BY season, posicao_prevista
"""
bq_run(view_top10_sql)
print("view gold.vw_top10_por_temporada criada.")


view gold.vw_top10_por_temporada criada.


In [ ]:
view_atual_sql = f"""
CREATE OR REPLACE VIEW `{PROJECT}.{GOLD}.vw_ranking_atual` AS
SELECT
  season, season_label, league, player, club_main, pos_primary,
  ROUND(predicted_ballon_dor_score, 2) AS score_previsto,
  RANK() OVER (PARTITION BY season ORDER BY predicted_ballon_dor_score DESC) AS posicao_prevista
FROM ML.PREDICT(MODEL `{PROJECT}.{GOLD}.model_btree_ballon_dor`,
  (SELECT {features_select}, season, season_label, player, club_main
   FROM `{PROJECT}.{GOLD}.player_season_features`
   WHERE split = 'inference'))
QUALIFY posicao_prevista <= 30
ORDER BY posicao_prevista
"""
bq_run(view_atual_sql)
print("view gold.vw_ranking_atual criada.")

if SEASONS_INFERENCIA:
    display(bq(f"SELECT * FROM `{PROJECT}.{GOLD}.vw_ranking_atual` LIMIT 10"))
else:
    print("nenhuma temporada sem rotulo disponivel ainda -- view fica vazia ate a Silver trazer uma.")


view gold.vw_ranking_atual criada.


,season,season_label,league,player,club_main,pos_primary,score_previsto,posicao_prevista
0,2425,2024-25,GER-Bundesliga,Harry Kane,Bayern Munich,FW,27.83,1
1,1920,2019-20,ESP-La Liga,Lionel Messi,Barcelona,FW,83.15,1
2,1920,2019-20,ENG-Premier League,Virgil van Dijk,Liverpool,DF,82.48,2
3,2425,2024-25,ENG-Premier League,Mohamed Salah,Liverpool,FW,27.21,2
4,1920,2019-20,ITA-Serie A,Cristiano Ronaldo,Juventus,FW,47.63,3
5,2425,2024-25,ESP-La Liga,Kylian Mbappé,Real Madrid,FW,23.12,3
6,2425,2024-25,FRA-Ligue 1,Ousmane Dembélé,Paris S-G,FW,19.91,4
7,1920,2019-20,ESP-La Liga,Sergio Ramos,Real Madrid,DF,18.71,4
8,2425,2024-25,ESP-La Liga,Robert Lewandowski,Barcelona,FW,16.91,5
9,1920,2019-20,FRA-Ligue 1,Neymar,Paris S-G,MF,17.20,5


## 12b. Previsão da temporada 2024-25 × Bola de Ouro 2025 real

A temporada 2024-25 ficou marcada como `inference` na Gold: o modelo nunca viu o resultado dela.
Como a Bola de Ouro 2025 já foi entregue, comparar a previsão com o ranking oficial é um teste
fora da amostra genuíno.

In [ ]:
SEASON_ALVO = 2425
assert SEASON_ALVO in SEASONS_INFERENCIA, "2024-25 não está marcada como inferência na Gold"

# recria a lista de features se a sessão foi reiniciada (sem retreinar nada)
if "features_select" not in globals():
    features_select = ",\n  ".join([f"{c}_z" for c in FEATURES_NUM] + FEATURES_CTX + FEATURES_CAT)

pred_2425_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT}.{GOLD}.predictions_2425` AS
SELECT
  season, season_label, league, player, club_main, pos_primary,
  ROUND(predicted_ballon_dor_score, 2) AS score_previsto,
  RANK() OVER (ORDER BY predicted_ballon_dor_score DESC) AS posicao_prevista
FROM ML.PREDICT(MODEL `{PROJECT}.{GOLD}.model_btree_ballon_dor`,
  (SELECT {features_select}, season, season_label, player, club_main
   FROM `{PROJECT}.{GOLD}.player_season_features`
   WHERE season = {SEASON_ALVO}))
"""
bq_run(pred_2425_sql)
print("gold.predictions_2425 materializada.")

pred_2425 = bq(f"SELECT * FROM `{PROJECT}.{GOLD}.predictions_2425` ORDER BY posicao_prevista")
print("\nTop-10 previsto para 2024-25:")
display(pred_2425.head(10)[["posicao_prevista", "player", "club_main", "league", "pos_primary", "score_previsto"]])

gold.predictions_2425 materializada.

Top-10 previsto para 2024-25:


,posicao_prevista,player,club_main,league,pos_primary,score_previsto
0,1,Harry Kane,Bayern Munich,GER-Bundesliga,FW,27.83
1,2,Mohamed Salah,Liverpool,ENG-Premier League,FW,27.21
2,3,Kylian Mbappé,Real Madrid,ESP-La Liga,FW,23.12
3,4,Ousmane Dembélé,Paris S-G,FRA-Ligue 1,FW,19.91
4,5,Robert Lewandowski,Barcelona,ESP-La Liga,FW,16.91
5,6,Abdoulaye Touré,Le Havre,FRA-Ligue 1,MF,16.47
6,7,Isco,Betis,ESP-La Liga,MF,13.20
7,8,Pedro,Lazio,ITA-Serie A,FW,12.69
8,9,Cole Palmer,Chelsea,ENG-Premier League,MF,11.86
9,10,Amir Rrahmani,Napoli,ITA-Serie A,DF,8.60


In [ ]:
# Bola de Ouro 2025 (temporada 2024-25) — ranking oficial, cerimônia de 22/09/2025
top10_real_2025 = pd.DataFrame({
    "rank_real": range(1, 11),
    "player": ["Ousmane Dembélé", "Lamine Yamal", "Vitinha", "Mohamed Salah", "Raphinha",
               "Achraf Hakimi", "Kylian Mbappé", "Cole Palmer", "Gianluigi Donnarumma", "Nuno Mendes"],
    "pontos_real": [1380, 1059, 703, 657, 620, 484, 378, 211, 172, 171],
})

comp = top10_real_2025.merge(
    pred_2425[["player", "club_main", "pos_primary", "posicao_prevista", "score_previsto"]],
    on="player", how="left")

print("Top-10 real de 2025 e onde o modelo colocou cada um:")
display(comp)

faltando = comp[comp.posicao_prevista.isna()].player.tolist()
if faltando:
    print("não encontrados na Gold (checar grafia):", faltando)

n_top10 = int((comp.posicao_prevista <= 10).sum())
n_top30 = int((comp.posicao_prevista <= 30).sum())
pos_vencedor = comp.loc[comp.rank_real == 1, "posicao_prevista"].iloc[0]

print(f"\nacertos no Top-10 (previsto × real)        : {n_top10}/10")
print(f"do Top-10 real, quantos no Top-30 previsto : {n_top30}/10")
print(f"posição prevista do vencedor (Dembélé)     : {int(pos_vencedor)}")
print(f"vencedor real no Top-10 previsto?          : {pos_vencedor <= 10}")

Top-10 real de 2025 e onde o modelo colocou cada um:


,rank_real,player,pontos_real,club_main,pos_primary,posicao_prevista,score_previsto
0,1,Ousmane Dembélé,1380,Paris S-G,FW,4,19.91
1,2,Lamine Yamal,1059,Barcelona,FW,18,4.69
2,3,Vitinha,703,Paris S-G,MF,76,0.44
3,4,Mohamed Salah,657,Liverpool,FW,2,27.21
4,5,Raphinha,620,Barcelona,FW,14,6.14
5,6,Achraf Hakimi,484,Paris S-G,DF,1504,-0.02
6,7,Kylian Mbappé,378,Real Madrid,FW,3,23.12
7,8,Cole Palmer,211,Chelsea,MF,9,11.86
8,9,Gianluigi Donnarumma,172,Paris S-G,GK,55,0.81
9,10,Nuno Mendes,171,Paris S-G,DF,271,0.03



acertos no Top-10 (previsto × real)        : 4/10
do Top-10 real, quantos no Top-30 previsto : 6/10
posição prevista do vencedor (Dembélé)     : 4
vencedor real no Top-10 previsto?          : True


## 13. Resumo da camada Gold

Rode a célula abaixo antes de fechar o notebook: é o slide de status final do pipeline completo
(Bronze → Silver → Gold → ML), para a apresentação do dia 11/09.

In [ ]:
print("=" * 66)
print("CAMADA GOLD -- RESUMO")
print("=" * 66)
print(f"tabela de features   : gold.player_season_features ({gold_tbl.num_rows:,} linhas)")
print(f"temporadas treino     : {SEASONS_TREINO}")
print(f"temporadas teste      : {SEASONS_TESTE}")
print(f"features no modelo    : {len(todas_features)} ({len(features_z)} numericas normalizadas + "
      f"{len(FEATURES_CTX)} contexto + {len(FEATURES_CAT)} categoricas)")
print()
print("modelos treinados:")
print("  - gold.model_linreg_ballon_dor         (LINEAR_REG, baseline)")
print("  - gold.model_btree_ballon_dor          (BOOSTED_TREE_REGRESSOR, principal)")
print("  - gold.model_btree_atk / mid / def      (extensao opcional, por posicao)")
print()
print("comparacao de metricas (teste, fora da amostra):")
display(comparacao[["modelo", "mean_absolute_error", "r2_score"]])
print()
print("overlap Top-10 / Top-30 por temporada de teste:")
display(resumo_ranking)
print()
print("views de BI:")
print("  - gold.vw_top10_por_temporada  (Top-10 previsto vs. real, por temporada de teste)")
print("  - gold.vw_ranking_atual        (ranking previsto para a temporada corrente, se disponivel)")
print("=" * 66)
print("\npipeline completo: bronze -> silver -> gold -> ML.PREDICT -> ranking Top-10")


CAMADA GOLD -- RESUMO
tabela de features   : gold.player_season_features (12,917 linhas)
temporadas treino     : [1718, 1819, 2021, 2122]
temporadas teste      : [2223, 2324]
features no modelo    : 37 (31 numericas normalizadas + 4 contexto + 2 categoricas)

modelos treinados:
  - gold.model_linreg_ballon_dor         (LINEAR_REG, baseline)
  - gold.model_btree_ballon_dor          (BOOSTED_TREE_REGRESSOR, principal)
  - gold.model_btree_atk / mid / def      (extensao opcional, por posicao)

comparacao de metricas (teste, fora da amostra):


,modelo,mean_absolute_error,r2_score
0,LINEAR_REG,0.990710,0.087442
1,BOOSTED_TREE_REGRESSOR,0.391534,0.209689



overlap Top-10 / Top-30 por temporada de teste:


,temporada,overlap_top10,overlap_top30,vencedor_real_no_top10_previsto
0,2022-23,2/10,10/30,True
1,2023-24,3/10,11/30,True



views de BI:
  - gold.vw_top10_por_temporada  (Top-10 previsto vs. real, por temporada de teste)
  - gold.vw_ranking_atual        (ranking previsto para a temporada corrente, se disponivel)

pipeline completo: bronze -> silver -> gold -> ML.PREDICT -> ranking Top-10


## 14. Apêndice — rodando tudo direto no BigQuery Console (sem Python)

Se a banca pedir para ver a etapa **só em SQL**, dentro do próprio BigQuery, este é o roteiro equivalente
ao que este notebook faz (troque `SEU_PROJETO` pelo ID do projeto e os números de temporada pelos valores impressos na §3):

```sql
-- 1) Tabela Gold (equivalente à secao 5) -------------------------------------
CREATE OR REPLACE TABLE `SEU_PROJETO.gold.player_season_features`
CLUSTER BY season, league AS
SELECT
  season, season_label, award_year, league, player_key, player, club_main,
  pos_primary, pos_raw, is_gk, age, minutes,
  CASE
    WHEN season IN (1718, 1819, 2021, 2122)   THEN 'train'   -- ajustar a sua lista real
    WHEN season IN (2223, 2324)               THEN 'test'    -- ajustar a sua lista real
    ELSE 'inference'
  END AS split,
  is_league_champion_proxy, is_ucl_spot_proxy, is_europe_spot_proxy,
  team_rank_proxy, team_ppm, team_ppm_on_pitch,
  SAFE_DIVIDE(goals_per90 - AVG(goals_per90) OVER (PARTITION BY season),
              NULLIF(STDDEV(goals_per90) OVER (PARTITION BY season), 0)) AS goals_per90_z,
  SAFE_DIVIDE(xg_per90 - AVG(xg_per90) OVER (PARTITION BY season),
              NULLIF(STDDEV(xg_per90) OVER (PARTITION BY season), 0)) AS xg_per90_z,
  SAFE_DIVIDE(assists_per90 - AVG(assists_per90) OVER (PARTITION BY season),
              NULLIF(STDDEV(assists_per90) OVER (PARTITION BY season), 0)) AS assists_per90_z,
  -- ... repetir SAFE_DIVIDE + AVG/STDDEV OVER (PARTITION BY season) para as demais
  --     features numericas listadas em FEATURES_NUM no notebook (secao 4) ...
  goals_per90, xg_per90, assists_per90, xag_per90, minutes, age,
  has_target, bd_rank, bd_points, ballon_dor_score, is_ballon_dor_ranked, is_top10, is_winner
FROM `SEU_PROJETO.silver.player_season`;

-- 2) Modelo principal (equivalente a secao 7) --------------------------------
CREATE OR REPLACE MODEL `SEU_PROJETO.gold.model_btree_ballon_dor`
OPTIONS(
  model_type = 'BOOSTED_TREE_REGRESSOR',
  input_label_cols = ['ballon_dor_score'],
  data_split_method = 'NO_SPLIT',
  max_iterations = 60, learn_rate = 0.08, max_tree_depth = 6,
  subsample = 0.85, l2_reg = 1.0, early_stop = TRUE
) AS
SELECT * EXCEPT(season, season_label, award_year, player_key, player, club_main,
                 pos_raw, split, has_target, bd_rank, bd_points, is_ballon_dor_ranked,
                 is_top10, is_winner)
FROM `SEU_PROJETO.gold.player_season_features`
WHERE split = 'train';

-- 3) Avaliacao (equivalente a secao 8) ----------------------------------------
SELECT *
FROM ML.EVALUATE(MODEL `SEU_PROJETO.gold.model_btree_ballon_dor`,
  (SELECT * FROM `SEU_PROJETO.gold.player_season_features` WHERE split = 'test'));

-- 4) Ranking Top-10 previsto vs. real (equivalente a secao 9) -----------------
SELECT
  player, club_main,
  ROUND(predicted_ballon_dor_score, 2) AS score_previsto,
  RANK() OVER (ORDER BY predicted_ballon_dor_score DESC) AS posicao_prevista
FROM ML.PREDICT(MODEL `SEU_PROJETO.gold.model_btree_ballon_dor`,
  (SELECT * FROM `SEU_PROJETO.gold.player_season_features`
   WHERE split = 'test' AND season = 2324))
ORDER BY posicao_prevista
LIMIT 10;

```

Este apêndice é só para consulta/defesa em SQL puro — o pipeline real deste notebook já roda tudo isso
via `client.query(...)` nas seções anteriores.